In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [2]:
from PIL import Image
from torchvision import transforms

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("koryakinp/fingers")

print("Path to dataset files:", path)

100%|██████████| 363M/363M [00:22<00:00, 17.0MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/koryakinp/fingers/versions/2


In [4]:
class FingerDataset(Dataset):
  def __init__(self, folder_path, transform = None):
    self.folder_path = folder_path
    self.transform = transform

    self.image_files = [f for f in os.listdir(folder_path)]

  def __len__(self):
    return len(self.image_files)

  def __getitem__(self, index):
    filename = self.image_files[index]

    label = int(filename.split('.')[0][-2])

    img_path = os.path.join(self.folder_path, filename)
    image = Image.open(img_path).convert('RGB')

    if self.transform:
      image = self.transform(image)

    return image, torch.tensor(label, dtype=torch.long)

In [5]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [6]:
test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [8]:
train_dataset = FingerDataset("/root/.cache/kagglehub/datasets/koryakinp/fingers/versions/2/train", transform=train_transform)
test_dataset = FingerDataset("/root/.cache/kagglehub/datasets/koryakinp/fingers/versions/2/test", transform=test_transform)

print(len(train_dataset))
print(len(test_dataset))

18000
3600


In [9]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [10]:
class ImageNeuralNetwork(nn.Module):
  def __init__(self,features_num=6):
   super().__init__()
   self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

   self.classifier = nn.Sequential(nn.Flatten(),
                                   nn.Linear(128*28*28,256),
                                   nn.ReLU(),
                                   nn.Dropout(0.5),
                                   nn.Linear(256,features_num))

  def forward(self,x):
    out = self.features(x)
    out = self.classifier(out)
    return out


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [12]:
model = ImageNeuralNetwork(features_num=6).to(device)

In [13]:
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [14]:
epochs = 25

In [15]:
for epoch in range(epochs):
  model.train()
  train_loss, correct = 0,0
  for features, labels in train_loader:
    images, labels = features.to(device), labels.to(device)

    optimizer.zero_grad()
    outputs = model(images)
    loss = loss_function(outputs, labels)
    loss.backward()
    optimizer.step()

    train_loss += loss.item()
    correct += (outputs.argmax(1) == labels).sum().item()
  print(f"Epoch: {epoch+1}, Loss: {train_loss}")
  train_acc = correct/len(train_dataset)*100
  print(f"Epoch: {epoch+1}, Train Accuracy: {train_acc}")


Epoch: 1, Loss: 84.13931217038044
Epoch: 1, Train Accuracy: 95.32222222222222
Epoch: 2, Loss: 11.539947826247953
Epoch: 2, Train Accuracy: 99.28888888888889
Epoch: 3, Loss: 4.911229104722629
Epoch: 3, Train Accuracy: 99.7388888888889
Epoch: 4, Loss: 5.430233291292524
Epoch: 4, Train Accuracy: 99.66666666666667
Epoch: 5, Loss: 4.303079371880436
Epoch: 5, Train Accuracy: 99.73333333333333
Epoch: 6, Loss: 5.527420944970267
Epoch: 6, Train Accuracy: 99.71111111111111
Epoch: 7, Loss: 0.590702530412113
Epoch: 7, Train Accuracy: 99.96666666666667
Epoch: 8, Loss: 8.374794232783138
Epoch: 8, Train Accuracy: 99.64444444444445
Epoch: 9, Loss: 2.3764637476736663
Epoch: 9, Train Accuracy: 99.87777777777778
Epoch: 10, Loss: 2.1783826410308276
Epoch: 10, Train Accuracy: 99.87777777777778
Epoch: 11, Loss: 1.6239434756675957
Epoch: 11, Train Accuracy: 99.91666666666667
Epoch: 12, Loss: 4.538709167630068
Epoch: 12, Train Accuracy: 99.81666666666666
Epoch: 13, Loss: 2.8858123620100313
Epoch: 13, Train Ac

In [18]:
model.eval()

for epoch in range(epochs):
    test_loss, test_correct = 0, 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            test_loss += loss_function(outputs, labels).item()
            test_correct += (outputs.argmax(1) == labels).sum().item()

    test_acc = test_correct / len(test_dataset) * 100

    print(f"Epoch {epoch+1:02d}/{epochs} | "
          f"Train Loss: {train_loss/len(train_loader):.4f} | "
          f"Train Acc: {train_acc:.1f}% | "
          f"Test Acc: {test_acc:.1f}%")

Epoch 01/25 | Train Loss: 0.0021 | Train Acc: 99.9% | Test Acc: 100.0%
Epoch 02/25 | Train Loss: 0.0021 | Train Acc: 99.9% | Test Acc: 100.0%
Epoch 03/25 | Train Loss: 0.0021 | Train Acc: 99.9% | Test Acc: 100.0%
Epoch 04/25 | Train Loss: 0.0021 | Train Acc: 99.9% | Test Acc: 100.0%
Epoch 05/25 | Train Loss: 0.0021 | Train Acc: 99.9% | Test Acc: 100.0%
Epoch 06/25 | Train Loss: 0.0021 | Train Acc: 99.9% | Test Acc: 100.0%
Epoch 07/25 | Train Loss: 0.0021 | Train Acc: 99.9% | Test Acc: 100.0%
Epoch 08/25 | Train Loss: 0.0021 | Train Acc: 99.9% | Test Acc: 100.0%
Epoch 09/25 | Train Loss: 0.0021 | Train Acc: 99.9% | Test Acc: 100.0%
Epoch 10/25 | Train Loss: 0.0021 | Train Acc: 99.9% | Test Acc: 100.0%
Epoch 11/25 | Train Loss: 0.0021 | Train Acc: 99.9% | Test Acc: 100.0%
Epoch 12/25 | Train Loss: 0.0021 | Train Acc: 99.9% | Test Acc: 100.0%
Epoch 13/25 | Train Loss: 0.0021 | Train Acc: 99.9% | Test Acc: 100.0%
Epoch 14/25 | Train Loss: 0.0021 | Train Acc: 99.9% | Test Acc: 100.0%
Epoch 

In [19]:
torch.save(model.state_dict(),'hand_sign.pth')

In [20]:
print(os.path.abspath('hand_sign.pth'))

/content/hand_sign.pth
